In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!kaggle datasets download kmader/skin-cancer-mnist-ham10000

Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:58<00:00, 94.7MB/s]



In [3]:
import numpy as np
import pandas as pd

In [6]:
# Interact with the Pyhton OS; File and Directory Operations.
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [7]:
# Data Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Training and Evaluating the Model
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import accuracy_score, classification_report

# TensorFlow for Deep Learning
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

In [8]:
import zipfile
with zipfile.ZipFile('skin-cancer-mnist-ham10000.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

In [9]:
import shutil

# Define the Paths to Source Directories

directory1 = '/content/HAM10000_images_part_1'
directory2 = '/content/HAM10000_images_part_2'

# Define the Path for the New Destination Directory

destination_directory = '/content/HAM10000_images'

# Create the Destination Directory
if not os.path.exists(destination_directory):
    os.makedirs(destination_directory)

# Function to Copy Files from Source directories to Destination Directory
def copy_files(source_dir, dest_dir):
    for filename in os.listdir(source_dir):
        source_path = os.path.join(source_dir, filename)
        destination_path = os.path.join(dest_dir, filename)
        if os.path.isfile(source_path):
            shutil.copy2(source_path, destination_path)

# Copy files from the First directory
copy_files(directory1, destination_directory)

# Copy files from the second directory
copy_files(directory2, destination_directory)
print("Directories have been Merged.")

Directories have been Merged.


In [10]:
# Load the Metadata
metadata = pd.read_csv('HAM10000_metadata.csv')
metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [11]:
# Load the Image Path
image_dir = './HAM10000_images'

# Adding the Image Path to the Corressponding Values
metadata['image_path'] = metadata['image_id'].apply(lambda x: os.path.join(image_dir, f'{x}.jpg'))
metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,image_path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,./HAM10000_images/ISIC_0027419.jpg
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,./HAM10000_images/ISIC_0025030.jpg
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,./HAM10000_images/ISIC_0026769.jpg
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,./HAM10000_images/ISIC_0025661.jpg
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,./HAM10000_images/ISIC_0031633.jpg


In [12]:
# Create an Object of ImageDataGenerator
datagen = ImageDataGenerator(
    preprocessing_function = preprocess_input,
    validation_split = 0.2,
    horizontal_flip = True,
    vertical_flip = True,
    zoom_range = 0.2,
    shear_range = 0.2,
    rotation_range = 20
)

In [13]:
# Defining the Target Image Size
target_size = (224, 224) # Resize Images to 224x224 — MobileNetV2 performs best at this input size.

# Batch Size is the number of Images processed at Once. This can be changed depending on the System Memory.
batch_size = 32

In [14]:
# Generating the Training Data
train_generator = datagen.flow_from_dataframe(
    dataframe = metadata,
    x_col = 'image_path',      # Feature Column (Image Data)
    y_col = 'dx',              # Target Column (Skin Cancer Classification)
    subset = "training",
    batch_size = batch_size,
    seed = 42,
    shuffle = True,
    class_mode = 'categorical',
    target_size = target_size
)

Found 8012 validated image filenames belonging to 7 classes.


In [15]:
# Generating the Validation Data
validation_generator = datagen.flow_from_dataframe(
    dataframe = metadata,
    x_col = 'image_path',      # Feature Column (Image Data)
    y_col = 'dx',              # Target Column (Skin Cancer Classification)
    subset = "validation",
    batch_size = batch_size,
    seed = 42,
    shuffle = True,
    class_mode = 'categorical',
    target_size = target_size
)

Found 2003 validated image filenames belonging to 7 classes.


In [16]:
# Calculating Class Weights to manage Class Imbalance
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weights = dict(enumerate(class_weights))

In [17]:
# Load the Base Model without Top Layer
base_model = MobileNetV2(input_shape = (224, 224, 3), weights = 'imagenet', include_top = False)
base_model.trainable = False  # Freeze Base Layers

# Compile the Custom Head Model on top of the Base Model
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.2)(x)
output = Dense(len(train_generator.class_indices), activation='softmax')(x)

model_mobilenet = Model(inputs=base_model.input, outputs=output)

# Compile the Custom Head Model on top of the Base Model
model_mobilenet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [18]:
model_mobilenet.fit(train_generator, validation_data=validation_generator, epochs=10, class_weight=class_weights)

Epoch 1/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 633s 2s/step - accuracy: 0.8354 - loss: 3.4353 - val_accuracy: 9.9850e-04 - val_loss: 7.1685
Epoch 2/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 623s 2s/step - accuracy: 0.8441 - loss: 2.1007 - val_accuracy: 9.9850e-04 - val_loss: 10.3335
Epoch 3/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 602s 2s/step - accuracy: 0.8465 - loss: 2.7436 - val_accuracy: 9.9850e-04 - val_loss: 10.0601
Epoch 4/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 633s 3s/step - accuracy: 0.8523 - loss: 1.9145 - val_accuracy: 9.9850e-04 - val_loss: 10.2597
Epoch 5/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 612s 2s/step - accuracy: 0.8570 - loss: 1.6651 - val_accuracy: 0.0015 - val_loss: 8.4037
Epoch 6/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 605s 2s/step - accuracy: 0.8646 - loss: 1.5004 - val_accuracy: 0.0125 - val_loss: 7.2778
Epoch 7/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 605s 2s/step - accuracy: 0.8684 - loss: 1.1588 - val_accuracy: 9.9850e-04 - val_loss: 10.2056
Epoch 8/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 614s 2s/step - accuracy: 0.8689 -

In [19]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.2, min_lr=1e-6),
    ModelCheckpoint('best_model.h5', save_best_only=True)
]

history = model_mobilenet.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=20,
    callbacks=callbacks
)


Epoch 1/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8751 - loss: 0.3800

251/251 ━━━━━━━━━━━━━━━━━━━━ 637s 3s/step - accuracy: 0.8802 - loss: 0.3524 - val_accuracy: 0.0090 - val_loss: 8.5135 - learning_rate: 0.0010
Epoch 2/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.8880 - loss: 0.3412

251/251 ━━━━━━━━━━━━━━━━━━━━ 631s 3s/step - accuracy: 0.8858 - loss: 0.3362 - val_accuracy: 0.0130 - val_loss: 8.4266 - learning_rate: 0.0010
Epoch 3/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 617s 2s/step - accuracy: 0.8903 - loss: 0.3209 - val_accuracy: 0.0045 - val_loss: 8.7957 - learning_rate: 0.0010
Epoch 4/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 614s 2s/step - accuracy: 0.8895 - loss: 0.3243 - val_accuracy: 0.0105 - val_loss: 8.8168 - learning_rate: 0.0010
Epoch 5/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 605s 2s/step - accuracy: 0.8907 - loss: 0.3141 - val_accuracy: 0.0075 - val_loss: 8.8174 - learning_rate: 2.0000e-04
Epoch 6/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 621s 2s/step - accuracy: 0.8959 - loss: 0.3094 - val_accuracy: 0.0075 - val_loss: 8.7564 - learning_rate: 2.0000e-04
Epoch 7/20
251/251 ━━━━━━━━━━━━━━━━━━━━ 633s 3s/step - accuracy: 0.8958 - loss: 0.2983 - val_accuracy: 0.0085 - val_loss: 8.8091 - learning_rate: 4.0000e-05


In [20]:
# Generating Predictions from the Model
y_true = validation_generator.classes
y_pred_proba = model_mobilenet.predict(validation_generator, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)

63/63 ━━━━━━━━━━━━━━━━━━━━ 122s 2s/step


In [21]:
 # Calculating Accuracy and Loss of the Model
val_loss, val_accuracy = model_mobilenet.evaluate(validation_generator, verbose=1)
print(f"\n✅ Validation Loss: {val_loss:.4f}")
print(f"✅ Validation Accuracy: {val_accuracy:.4f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 123s 2s/step - accuracy: 0.0170 - loss: 8.4622

✅ Validation Loss: 8.4622
✅ Validation Accuracy: 0.0170
